# Análisis de inversionistas| Crunchbase BI
**Rama base:** `hugo/fred` · **Fuente:** exclusivamente `marts` · **Corte:** 2013-12-31 · **Cobertura principal:** rondas desde 2005.

Las métricas se calculan desde `marts.puente_ronda_inversionista`, `marts.fact_rondas`, `marts.dim_inversionista`, `marts.dim_empresa` y `marts.met_salida_empresa`. Se guardan **tres tablas de resultados** en DuckDB para su publicación posterior. No se generan CSV de resultados.

## 1. Conexión y validación de insumos

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run_all.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abrí el notebook desde el repositorio que contiene run_all.py")
DB = ROOT / "data" / "crunchbase.duckdb"
if not DB.exists():
    raise FileNotFoundError(f"Falta {DB}. Ejecutá primero python run_all.py desde la raíz con los datos raw instalados.")
con = duckdb.connect(str(DB))
requeridas = {
    'puente_ronda_inversionista': {'ronda_id','inversionista_id','peso'},
    'fact_rondas': {'ronda_id','empresa_id','fecha','monto_usd'},
    'dim_inversionista': {'inversionista_id','nombre','tipo_inversionista','pais'},
    'dim_empresa': {'empresa_id','industria','pais','etapa'},
    'met_salida_empresa': {'empresa_id','fecha_primera_salida','tuvo_salida'},
}
for tabla, columnas in requeridas.items():
    reales = {r[0] for r in con.execute(f'DESCRIBE marts.{tabla}').fetchall()}
    faltantes = columnas - reales
    if faltantes:
        raise ValueError(f'Faltan columnas en marts.{tabla}: {sorted(faltantes)}')
print('Esquema marts validado; DuckDB:', DB)

Esquema marts validado; DuckDB: C:\Users\Nany\proyectofinal\Proyecto_BI_Ximena_FINAL\crunchbase-bi-proyecto-final-ximena-integrado\data\crunchbase.duckdb


## 2. Reglas y denominadores
- **Universo principal:** participaciones en rondas fechadas desde `2005-01-01` hasta `2013-12-31`.
- **Inversionista activo:** al menos una participación en ese universo. `n_rondas` cuenta rondas distintas; `n_empresas` cuenta empresas distintas.
- **Éxito V1:** proporción de empresas únicas del portafolio con salida (IPO o adquisición) **posterior o igual a la primera inversión observada del inversionista**, hasta el cierre de 2013.
- **Éxito V2:** proporción de empresas únicas con salida observada **dentro de 5 años** de la primera inversión; solo se incluyen empresas cuya primera inversión ocurrió a más tardar en `2008-12-31` para observar la ventana completa. El denominador V2 puede ser menor al V1.
- **Benchmark:** tasa V1 de cada inversionista frente a la tasa V1 del grupo de pares de su `tipo_inversionista`, contando una empresa una vez por inversionista. Diferencia en puntos porcentuales. Rankings con al menos 5 empresas.
- El **monto atribuido** es `monto_usd * peso`, una aproximación de reparto igualitario, **no dinero realmente aportado**. Montos desconocidos permanecen NULL. La tasa de éxito no es retorno financiero ni causalidad.

## 3. Tablas de resultados (se crean en `marts`)

In [2]:
con.execute("""
CREATE OR REPLACE TABLE marts.res_inversionista_activo AS
WITH participaciones AS (
 SELECT DISTINCT p.inversionista_id, p.ronda_id, r.empresa_id, r.fecha, r.monto_usd, p.peso
 FROM marts.puente_ronda_inversionista p
 JOIN marts.fact_rondas r ON p.ronda_id = r.ronda_id
 WHERE r.fecha BETWEEN DATE '2005-01-01' AND DATE '2013-12-31'
   AND r.empresa_id IS NOT NULL
)
SELECT i.inversionista_id, i.nombre, i.tipo_inversionista, i.pais,
 COUNT(DISTINCT p.ronda_id) AS n_rondas,
 COUNT(DISTINCT p.empresa_id) AS n_empresas,
 MIN(p.fecha) AS primera_fecha_observada,
 MAX(p.fecha) AS ultima_fecha_observada,
 SUM(CASE WHEN p.monto_usd IS NOT NULL THEN p.monto_usd * p.peso END)::DOUBLE AS monto_atribuido_usd,
 COUNT(DISTINCT CASE WHEN p.monto_usd IS NOT NULL THEN p.ronda_id END) AS rondas_con_monto,
 TRUE AS activo_2005_2013
FROM participaciones p JOIN marts.dim_inversionista i USING (inversionista_id)
GROUP BY i.inversionista_id, i.nombre, i.tipo_inversionista, i.pais
""")
print('Activos:',con.execute('SELECT COUNT(*) FROM marts.res_inversionista_activo').fetchone()[0])

Activos: 16897


In [3]:
con.execute("""
CREATE OR REPLACE TABLE marts.res_exito_inversionista AS
WITH primera AS (
 SELECT p.inversionista_id, r.empresa_id, MIN(r.fecha) AS primera_inversion
 FROM marts.puente_ronda_inversionista p
 JOIN marts.fact_rondas r ON p.ronda_id = r.ronda_id
 WHERE r.fecha BETWEEN DATE '2005-01-01' AND DATE '2013-12-31'
   AND r.empresa_id IS NOT NULL
 GROUP BY p.inversionista_id, r.empresa_id
), evaluacion AS (
 SELECT a.*, s.fecha_primera_salida,
  (s.fecha_primera_salida BETWEEN a.primera_inversion AND DATE '2013-12-31') AS exito_v1,
  (a.primera_inversion <= DATE '2008-12-31') AS elegible_v2,
  (s.fecha_primera_salida BETWEEN a.primera_inversion AND a.primera_inversion + INTERVAL '5 years') AS exito_v2
 FROM primera a LEFT JOIN marts.met_salida_empresa s USING (empresa_id)
)
SELECT e.inversionista_id,
 COUNT(*) AS n_empresas,
 COUNT(*) FILTER (WHERE COALESCE(exito_v1,FALSE)) AS n_exitos_v1,
 (COUNT(*) FILTER (WHERE COALESCE(exito_v1,FALSE)) * 1.0 / NULLIF(COUNT(*),0))::DOUBLE AS tasa_v1,
 COUNT(*) FILTER (WHERE elegible_v2) AS n_elegibles_v2,
 COUNT(*) FILTER (WHERE elegible_v2 AND COALESCE(exito_v2,FALSE)) AS n_exitos_v2,
 (COUNT(*) FILTER (WHERE elegible_v2 AND COALESCE(exito_v2,FALSE)) * 1.0 /
  NULLIF(COUNT(*) FILTER (WHERE elegible_v2),0))::DOUBLE AS tasa_v2
FROM evaluacion e GROUP BY e.inversionista_id
""")
display(con.execute('SELECT * FROM marts.res_exito_inversionista ORDER BY n_empresas DESC LIMIT 10').df())

,inversionista_id,n_empresas,n_exitos_v1,tasa_v1,n_elegibles_v2,n_exitos_v2,tasa_v2
0,c:2520,459,64,0.139434,94,17,0.180851
1,f:3181,435,108,0.248276,65,21,0.323077
2,f:367,377,71,0.188329,180,37,0.205556
3,f:316,319,56,0.175549,126,23,0.182540
4,f:5928,315,31,0.098413,0,0,NaN
5,f:17,297,68,0.228956,132,36,0.272727
6,f:4,284,60,0.211268,111,25,0.225225
7,c:362,268,29,0.108209,19,7,0.368421
8,f:26,268,55,0.205224,150,35,0.233333
9,f:41,267,42,0.157303,89,17,0.191011


In [4]:
con.execute("""
CREATE OR REPLACE TABLE marts.res_benchmark_inversionista AS
WITH pares AS (
 SELECT a.tipo_inversionista,
 SUM(e.n_exitos_v1) * 1.0 / NULLIF(SUM(e.n_empresas),0) AS tasa_v1_pares,
 COUNT(*) AS n_inversionistas_pares
 FROM marts.res_exito_inversionista e
 JOIN marts.res_inversionista_activo a USING (inversionista_id)
 GROUP BY a.tipo_inversionista
)
SELECT a.inversionista_id, a.nombre, a.tipo_inversionista, a.pais,
 a.n_rondas, e.n_empresas, e.n_exitos_v1, e.tasa_v1,
 e.n_elegibles_v2, e.n_exitos_v2, e.tasa_v2,
 p.tasa_v1_pares::DOUBLE AS benchmark_tasa_v1,
 ((e.tasa_v1 - p.tasa_v1_pares) * 100)::DOUBLE AS diferencia_pp,
 p.n_inversionistas_pares,
 (e.n_empresas >= 5) AS elegible_ranking
FROM marts.res_inversionista_activo a
JOIN marts.res_exito_inversionista e USING (inversionista_id)
JOIN pares p USING (tipo_inversionista)
""")
display(con.execute("""SELECT nombre, tipo_inversionista, n_empresas, n_exitos_v1,
 ROUND(tasa_v1*100,2) AS exito_v1_pct, ROUND(tasa_v2*100,2) AS exito_v2_pct,
 ROUND(diferencia_pp,2) AS diferencia_pp
 FROM marts.res_benchmark_inversionista WHERE elegible_ranking
 ORDER BY tasa_v1 DESC, n_empresas DESC LIMIT 10""").df())

,nombre,tipo_inversionista,n_empresas,n_exitos_v1,exito_v1_pct,exito_v2_pct,diferencia_pp
0,Merchant Banking Devision,organizacion_financiera,9,8,88.89,75.00,73.55
1,i-Hatch Ventures,organizacion_financiera,5,4,80.00,75.00,64.66
2,NEC,corporativo,5,4,80.00,75.00,70.12
3,Brian Pokorny,persona,12,9,75.00,50.00,62.19
4,TD Capital,organizacion_financiera,11,8,72.73,70.00,57.39
5,Samsung Ventures America,organizacion_financiera,7,5,71.43,60.00,56.09
6,Smedvig Capital,organizacion_financiera,12,8,66.67,88.89,51.33
7,DFJ Portage Ventures,organizacion_financiera,9,6,66.67,75.00,51.33
8,Li Ka-shing,persona,6,4,66.67,66.67,53.86
9,Israel Seed Partners,organizacion_financiera,6,4,66.67,50.00,51.33


## 4. Validaciones y comparación de cobertura histórica

In [5]:
for tabla in ['res_inversionista_activo','res_exito_inversionista','res_benchmark_inversionista']:
    n = con.execute(f'SELECT COUNT(*) FROM marts.{tabla}').fetchone()[0]
    dup = con.execute(f'SELECT COUNT(*) FROM (SELECT inversionista_id FROM marts.{tabla} GROUP BY 1 HAVING COUNT(*) > 1)').fetchone()[0]
    assert dup == 0, tabla
    print(tabla, n, 'filas, sin duplicados')
assert con.execute('SELECT COUNT(*) FROM marts.res_exito_inversionista WHERE n_exitos_v1 > n_empresas OR n_exitos_v2 > n_elegibles_v2').fetchone()[0] == 0
print('Validación básica OK')


res_inversionista_activo 16897 filas, sin duplicados
res_exito_inversionista 16897 filas, sin duplicados
res_benchmark_inversionista 16897 filas, sin duplicados
Validación básica OK


## 5. Publicación y limitaciones

Las tablas `marts.res_inversionista_activo`, `marts.res_exito_inversionista` y `marts.res_benchmark_inversionista` se guardan **en la base DuckDB local**. Ejecutar después `python publish/publish_to_neon.py` desde la raíz para publicarlas; el publicador necesita que las tres tablas estén declaradas en `PKS` (se incluye el ajuste en el paquete).


**Limitaciones:** Crunchbase tiene subcobertura anterior a 2005; las empresas sin salida a 2013 están censuradas; V1 favorece inversiones antiguas con más tiempo de observación; V2 restringe el denominador a inversiones hasta 2008; una salida no prueba rentabilidad; no se conocen los aportes reales por inversionista; las salidas anteriores a la inversión no cuentan como éxito del inversionista.